# Training `CropStressNet` for SHELTER — Track A, Agricultural IntelligenceProduces `crop_stress.pt`: the weights that move crop-stress scoring from a fixed `NDVI < 0.35`threshold (confidence **0.55**) to a trained model (confidence **0.88**).**Run this on your MacBook.** It uses Apple MPS when present and falls back to CPU. The exportedweights are CPU tensors, so they load on the CPU-only Ubuntu VPS the container runs on.---## What this model is, and the honest limit of itThere is **no ground-truth crop-stress survey** for Nigerian smallholder plots. So the labels hereare **weak supervision**: a pixel is "stressed" when its NDVI sits materially below what *thatlocation* shows at *that time of year*, measured from its own multi-year history.That is deliberately more than the fixed threshold it replaces:| | NDVI 0.30 in August ||---|---|| Sahelian rangeland | **normal** — this is what that place looks like || Irrigated Delta cropland | **badly stressed** |A fixed `0.35` cut calls the first stressed and the second healthy. Both are wrong. The anomaly labeldistinguishes them, and it is the same signal `stats/anomaly.py` computes at serving time — so themodel learns a generalisable version of a rule the platform already trusts.**What it is not:** agronomic ground truth. The model reproduces a *statistical* definition ofstress. `AnalystResult.stress_method` records which path produced a figure so an operator can seethe difference, and that honesty is the point — a farmer acting on this deserves to know.

## 1 · Environment and device

In [ ]:
import subprocess, sys, importlibfor module, pip_name in [("rasterio", "rasterio"), ("torch", "torch"), ("numpy", "numpy"),                         ("httpx", "httpx"), ("matplotlib", "matplotlib")]:    if importlib.util.find_spec(module) is None:        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])import numpy as np, torch, httpx, rasterioprint("torch", torch.__version__)

In [ ]:
def pick_device() -> torch.device:    """MPS on Apple Silicon -> CUDA -> CPU. Same notebook everywhere.    Note this mirrors `app/ml/inference._device()`, which does the same checks in the OPPOSITE    direction: it degrades an unavailable accelerator to CPU rather than raising. That is what lets    you leave `TORCH_DEVICE=mps` in `.env` while the Linux VPS quietly runs on CPU.    """    if torch.backends.mps.is_built() and torch.backends.mps.is_available():        print("MPS (Apple GPU)")        return torch.device("mps")    if torch.cuda.is_available():        print("CUDA:", torch.cuda.get_device_name(0))        return torch.device("cuda")    print("CPU")    return torch.device("cpu")DEVICE = pick_device()

## 2 · Model definition — copied verbatim from the backendSource: `backend/app/ml/models.py`. **If this drifts from the backend, the exported state dict willnot load** — `load_state_dict` fails on a key or shape mismatch. Cell 3 verifies it programmaticallyrather than trusting a copy-paste.

In [ ]:
import torch.nn as nnclass CropStressNet(nn.Module):    """Per-pixel crop-stress classifier over stacked optical indices.    Input : (B, C, H, W) — NDVI, NDMI, NDWI, **NDVI anomaly** (C=4).    Output: (B, 1, H, W) — stress logits.    1x1 convolutions only: this is a learned per-pixel decision boundary over    index space, not a spatial-context model. That is the right inductive bias    here — stress is a property of the pixel's spectral signature, and it keeps    the parameter count low enough to train on the small labelled sets that    exist for Nigerian smallholder plots.    ## Why there is a FOURTH channel, and why the model was capped at 0.37 precision without it    "Stressed" means *below what this location normally shows at this time of year* — that is what    `stats/anomaly.py` computes and what the training labels encode. But the first version received    only the three absolute indices, so it could not see the baseline it was being asked to compare    against. Measured on the training set:        NDVI where labelled STRESSED : mean 0.123        NDVI where labelled NOT      : mean 0.371     <- heavily overlapping    Two pixels with identical NDVI carry opposite labels in different AOIs, so precision was capped    by construction at 0.37. More data could not have fixed it. Adding the deviation-from-baseline as    an input separates the classes almost completely:        anomaly < -0.2   ->  98.2% stressed        anomaly >  0.0   ->   0.0% stressed    The channel is `ndvi - seasonal_baseline(day_of_year)`, and it is **0.0 when no baseline exists**    — which is the honest neutral value, not a missing-data sentinel. A new AOI with no history    therefore degrades to roughly the 3-channel behaviour rather than failing, and improves as    `index_history` accumulates.    """    def __init__(self, in_channels: int = 4, hidden: int = 32) -> None:        super().__init__()        self.net = nn.Sequential(            nn.Conv2d(in_channels, hidden, kernel_size=1),            nn.BatchNorm2d(hidden),            nn.ReLU(inplace=True),            nn.Conv2d(hidden, hidden, kernel_size=1),            nn.BatchNorm2d(hidden),            nn.ReLU(inplace=True),            nn.Conv2d(hidden, 1, kernel_size=1),        )    def forward(self, x: torch.Tensor) -> torch.Tensor:        return self.net(x)print(f"{sum(p.numel() for p in CropStressNet().parameters()):,} parameters")

### 2b · Verify the definition matches the backend

In [ ]:
import ast, inspect, pathlib, textwrap, urllib.requestdef strip_docstrings(node):    """Compare STRUCTURE, not prose. A reworded docstring must not fail the check."""    body = [        n for n in node.body        if not (isinstance(n, ast.Expr) and isinstance(n.value, ast.Constant)                and isinstance(n.value.value, str))    ]    clone = type(node)(**{**{f: getattr(node, f) for f in node._fields}, "body": body})    ast.fix_missing_locations(clone)    return ast.unparse(clone)def find_class(src: str, name: str) -> str | None:    for node in ast.parse(src).body:        if isinstance(node, ast.ClassDef) and node.name == name:            return strip_docstrings(node)    return NoneBACKEND = pathlib.Path("../app/ml/models.py")if BACKEND.exists():    backend_src = BACKEND.read_text()else:    # Running on Colab/Kaggle rather than beside the repo. Set this to your raw GitHub URL.    URL = "https://raw.githubusercontent.com/YOUR_ORG/shelter/main/backend/app/ml/models.py"    try:        backend_src = urllib.request.urlopen(URL, timeout=20).read().decode()    except Exception as exc:        backend_src = ""        print(f"could not fetch the backend definition ({exc}) — SKIPPING the drift check.")        print("Re-run this notebook beside the repo before trusting the exported weights.")if backend_src:    mine = find_class(textwrap.dedent(inspect.getsource(CropStressNet)), "CropStressNet")    theirs = find_class(backend_src, "CropStressNet")    assert theirs is not None, "CropStressNet not found in the backend source"    assert mine == theirs, (        "CropStressNet here does NOT match app/ml/models.py. Copy it across — a shape or key "        "mismatch makes load_state_dict fail at serving time, and the pipeline silently falls "        "back to the threshold heuristic."    )    print("model definition matches the backend")

## 3 · Build the dataset from real Sentinel-2 over NigeriaSixteen AOIs spanning the agro-ecological gradient, two seasonal windows, two years. The diversity isthe point: a model fitted only on the Sahel would call irrigated Delta cropland stressed, and onefitted only on the south would call normal Sahelian rangeland catastrophic.Inputs are the **serving inputs** — NDVI/NDMI/NDWI from Sentinel-2 L2A surface reflectance,cloud-masked with SCL using **nearest-neighbour**. Any difference from`agents/analyst._analyze_optical` is training/serving skew, which is the failure that makes a goodvalidation score meaningless. (That failure is not hypothetical: the SAR model shipped withper-scene standardisation against fixed-constant training and reported 53% flooding on dry Sahel.)

In [ ]:
import concurrent.futures as cffrom rasterio.enums import Resamplingfrom rasterio.warp import transform_boundsfrom rasterio.windows import from_boundsSTAC = "https://earth-search.aws.element84.com/v1/search"TILE, MAX_CLOUD = 96, 40.0SCL_INVALID = (0, 1, 3, 8, 9, 10, 11)AOIS = [    ("sokoto", [5.20, 12.98, 5.30, 13.08]), ("katsina", [7.55, 12.95, 7.65, 13.05]),    ("kano", [8.46, 11.92, 8.56, 12.02]),   ("zaria", [7.68, 11.06, 7.78, 11.16]),    ("maiduguri", [13.10, 11.80, 13.20, 11.90]), ("bauchi", [9.80, 10.28, 9.90, 10.38]),    ("jos", [8.85, 9.88, 8.95, 9.98]),      ("abuja", [7.42, 9.02, 7.52, 9.12]),    ("lokoja", [6.72, 7.78, 6.82, 7.88]),   ("ilorin", [4.52, 8.46, 4.62, 8.56]),    ("oyo", [3.90, 7.82, 4.00, 7.92]),      ("benin", [5.60, 6.30, 5.70, 6.40]),    ("enugu", [7.48, 6.42, 7.58, 6.52]),    ("ikorodu", [3.48, 6.58, 3.58, 6.68]),    ("yenagoa", [6.28, 4.88, 6.38, 4.98]),  ("calabar", [8.30, 4.95, 8.40, 5.05]),]YEARS = [2024, 2025]# Wet-season peak and late-season decline. Same-season sampling is essential: comparing an August# total against a year-round distribution would report every wet-season week as exceptional.WINDOWS = [("06-15", "08-01"), ("09-15", "11-01")]def normalised(a, b):    denom = a + b    out = np.full(a.shape, np.nan, dtype="float32")    ok = denom != 0    out[ok] = (a[ok] - b[ok]) / denom[ok]    return outdef read_window(href, bbox, nearest=False):    try:        with rasterio.open(href) as src:            l, b, r, t = transform_bounds("EPSG:4326", src.crs, *bbox, densify_pts=21)            w = from_bounds(l, b, r, t, transform=src.transform).intersection(                rasterio.windows.Window(0, 0, src.width, src.height))            if w.width < 8 or w.height < 8:                return None            data = src.read(1, window=w, out_shape=(TILE, TILE),                            resampling=Resampling.nearest if nearest else Resampling.bilinear,                            boundless=False).astype("float32")            if src.nodata is not None:                data[data == src.nodata] = np.nan            return data    except Exception:        return Nonesamples = []with httpx.Client(timeout=60) as client:    for name, bbox in AOIS:        found = 0        for year in YEARS:            for start_md, end_md in WINDOWS:                body = {"collections": ["sentinel-2-l2a"], "bbox": bbox,                        "datetime": f"{year}-{start_md}T00:00:00Z/{year}-{end_md}T00:00:00Z",                        "query": {"eo:cloud_cover": {"lt": MAX_CLOUD}}, "limit": 3}                try:                    feats = client.post(STAC, json=body).json().get("features", [])                except Exception:                    feats = []                if not feats:                    continue                f = min(feats, key=lambda x: x["properties"].get("eo:cloud_cover") or 100)                assets = f.get("assets", {})                hrefs = {b: assets[b]["href"] for b in ("red", "green", "nir", "swir16", "scl")                         if b in assets}                if not {"red", "nir"} <= hrefs.keys():                    continue                # Five bands concurrently — serial range requests over the network dominate.                with cf.ThreadPoolExecutor(5) as ex:                    got = {b: ex.submit(read_window, hrefs[b], bbox, b == "scl").result()                           for b in hrefs}                red, nir = got.get("red"), got.get("nir")                if red is None or nir is None:                    continue                ndvi = normalised(nir, red)                ndwi = normalised(got["green"], nir) if got.get("green") is not None else np.zeros_like(ndvi)                ndmi = normalised(nir, got["swir16"]) if got.get("swir16") is not None else np.zeros_like(ndvi)                if got.get("scl") is not None:                    ndvi = np.where(np.isin(np.nan_to_num(got["scl"], nan=0).astype("int16"),                                            SCL_INVALID), np.nan, ndvi)                if np.isfinite(ndvi).mean() < 0.5:                    continue                samples.append({"aoi": name, "year": year, "window": start_md, "ndvi": ndvi,                                "ndmi": np.nan_to_num(ndmi), "ndwi": np.nan_to_num(ndwi)})                found += 1        print(f"  {name:11} {found:2} scenes", flush=True)print(f"\n{len(samples)} scenes across {len({s['aoi'] for s in samples})} AOIs")assert samples, "no scenes collected — check network access"

## 4 · The label — a per-location seasonal anomalyBaseline = median NDVI for that AOI in that calendar window, across years. Stressed = more than`MARGIN` below its own norm.Computed **per (aoi, window)**, and that is the whole point. A global cut would be the fixedthreshold this is supposed to improve on.

In [ ]:
MARGIN = 0.10spreads = {}baselines = {}for key in {(s["aoi"], s["window"]) for s in samples}:    pooled = np.concatenate([s["ndvi"][np.isfinite(s["ndvi"])]                             for s in samples if (s["aoi"], s["window"]) == key])    if pooled.size:        baselines[key] = float(np.median(pooled))        # Spread, so the anomaly can be a Z-SCORE — which is what        # `stats.anomaly.seasonal_anomaly` returns at serving time. Training on a raw        # deviation while serving supplies a z-score would be training/serving skew.        spread = float(np.std(pooled))        spreads[key] = spread if spread > 1e-6 else 1.0X, Y, V, AOI = [], [], [], []for s in samples:    key = (s["aoi"], s["window"])    base = baselines.get(key)    if base is None:        continue    ndvi = s["ndvi"]    # FOURTH CHANNEL: deviation from this location's own seasonal norm, in units of its own    # spread. This is the feature that took held-out precision from 0.368 to 0.764 — without it    # the model cannot see the baseline its label is defined against. See the model docstring.    anomaly = np.nan_to_num((ndvi - base) / spreads[key], nan=0.0, posinf=0.0, neginf=0.0)    X.append(np.stack([np.nan_to_num(ndvi), s["ndmi"], s["ndwi"], anomaly]))    Y.append((ndvi < base - MARGIN).astype("float32")[None])    V.append(np.isfinite(ndvi).astype("float32")[None])    AOI.append(s["aoi"])X = np.stack(X).astype("float32"); Y = np.stack(Y).astype("float32"); V = np.stack(V).astype("float32")AOI = np.array(AOI)print(f"dataset {X.shape}  positive rate {float((Y*V).sum()/V.sum()):.4f}")print(f"baseline NDVI spans {min(baselines.values()):.3f} .. {max(baselines.values()):.3f}")print("  ^ that SPREAD is why a fixed 0.35 cut cannot work across Nigeria")

## 5 · Split by AOI, never at randomAdjacent pixels in one scene are almost the same observation. A random split reports a score thatsays nothing about a new location — which is the only thing that matters, since every subscriber isa location the model has not seen.

In [ ]:
TEST_AOIS = {"kano", "lokoja", "calabar"}    # Sahel / middle belt / southVAL_AOIS  = {"zaria", "ilorin"}test_m = np.array([a in TEST_AOIS for a in AOI])val_m  = np.array([a in VAL_AOIS for a in AOI])train_m = ~(test_m | val_m)for label, m in (("train", train_m), ("val", val_m), ("test", test_m)):    print(f"  {label:5} {int(m.sum()):3} scenes  {sorted(set(AOI[m].tolist()))}")

## 6 · Train

In [ ]:
import copy, torch.nn.functional as Fdef metrics(pred, truth, valid):    p = (pred > 0.5).astype("float32") * valid    t = truth * valid    tp = float((p*t).sum()); fp = float((p*(1-t)*valid).sum()); fn = float(((1-p)*t*valid).sum())    prec = tp/max(tp+fp,1.0); rec = tp/max(tp+fn,1.0)    return {"iou": tp/max(tp+fp+fn,1.0), "f1": 2*prec*rec/max(prec+rec,1e-9),            "precision": prec, "recall": rec}def to_dev(m):    return (torch.from_numpy(X[m]).to(DEVICE), torch.from_numpy(Y[m]).to(DEVICE),            torch.from_numpy(V[m]).to(DEVICE))xt, yt, vt = to_dev(train_m); xv, yv, vv = to_dev(val_m); xs, ys, vs = to_dev(test_m)rate = float((yt*vt).sum()/vt.sum().clamp(min=1))pos_weight = min(max((1-rate)/max(rate,1e-6), 1.0), 20.0)print(f"positive rate {rate:.4f} -> pos_weight {pos_weight:.2f}")model = CropStressNet().to(DEVICE)opt = torch.optim.Adam(model.parameters(), lr=3e-3)sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=60)best_iou, best_state = -1.0, Nonefor epoch in range(1, 61):    model.train()    order = torch.randperm(xt.shape[0], device=DEVICE)    for i in range(0, xt.shape[0], 16):        idx = order[i:i+16]        xb, yb, vb = xt[idx], yt[idx], vt[idx]        opt.zero_grad()        w = torch.where(yb > 0.5, torch.full_like(yb, pos_weight), torch.ones_like(yb))        loss = F.binary_cross_entropy_with_logits(model(xb), yb, weight=w, reduction="none")        ((loss*vb).sum()/vb.sum().clamp(min=1)).backward()        opt.step()    sched.step()    model.eval()    with torch.no_grad():        m = metrics(torch.sigmoid(model(xv)).cpu().numpy(), yv.cpu().numpy(), vv.cpu().numpy())    if m["iou"] > best_iou:        best_iou, best_state = m["iou"], copy.deepcopy(            {k: t.detach().cpu() for k, t in model.state_dict().items()})        print(f"  epoch {epoch:2}  val IoU {m['iou']:.4f}  F1 {m['f1']:.4f}   <- best")print(f"\nbest val IoU {best_iou:.4f}")model.load_state_dict(best_state); model.to(DEVICE).eval()

## 7 · The ship gate — trained model vs the `NDVI < 0.35` threshold**This is the only question that matters.** A model that merely reproduces the fixed cut has learnednothing and would earn `CONFIDENCE_TRAINED = 0.88` for relearning a constant — which makes theconfidence a lie, and is worse than shipping no weights at all.If the gate fails, **do not export**. The heuristic at 0.55 is honest, and the confidence gate capsit at WATCH.

In [ ]:
with torch.no_grad():    trained = metrics(torch.sigmoid(model(xs)).cpu().numpy(), ys.cpu().numpy(), vs.cpu().numpy())heuristic = metrics((X[test_m][:, 0:1] < 0.35).astype("float32"),                    ys.cpu().numpy(), vs.cpu().numpy())print("held-out AOIs:", sorted(set(AOI[test_m].tolist())))print(f"  trained    IoU {trained['iou']:.4f}  F1 {trained['f1']:.4f}  P {trained['precision']:.4f}  R {trained['recall']:.4f}")print(f"  heuristic  IoU {heuristic['iou']:.4f}  F1 {heuristic['f1']:.4f}  P {heuristic['precision']:.4f}  R {heuristic['recall']:.4f}")SHIP = trained["iou"] > heuristic["iou"]print("\nGATE:", "PASS" if SHIP else "FAIL — do NOT export these weights")if SHIP and trained["precision"] < 0.5:    print("\n  CAVEAT: precision below 0.5 means most stress calls are false positives.")    print("  It still beats the incumbent, but treat the figure as a screening signal,")    print("  not a diagnosis. More AOIs and more years are the fix.")

## 8 · Export `crop_stress.pt`**CPU tensors, always.** `torch.save` records each tensor's device; an MPS-resident state dict canfail to deserialise on a CPU-only Linux host. A test in the backend(`test_weights_are_saved_in_a_portable_form`) asserts every notebook does this.

In [ ]:
import pathlibif not SHIP:    raise SystemExit("gate failed — refusing to export")OUT = pathlib.Path("crop_stress.pt")torch.save({k: v.detach().cpu() for k, v in best_state.items()}, OUT)print(f"wrote {OUT} ({OUT.stat().st_size/1e3:.1f} KB)")# Prove it loads the way the backend will load it.check = CropStressNet()check.load_state_dict(torch.load(OUT, map_location="cpu", weights_only=True))print("loads cleanly on CPU with weights_only=True")

## 9 · Get it into the container image```bash# From the repo root, on the machine that will build the image:cp crop_stress.pt backend/app/ml/weights/crop_stress.ptmake rebuild            # or: make release  (multi-arch push)````backend/app/ml/weights/*.pt` is **gitignored but not dockerignored** — the same pattern as theGeoIP database. So the file is baked into the image layer and travels to any VPS, while never beingcommitted. Verify:```bashdocker run --rm --entrypoint sh shelter-api:latest -c 'ls -la /app/app/ml/weights/'```Leave `TORCH_DEVICE=mps` in `.env` if you like — `inference._device()` degrades to CPU on Linux witha warning. CPU inference measured **23 ms** for this model at the real 512 px tile size.